# Market Share & Vertical Benchmarking — Market Share & Vertical Benchmarking

Build a **grounded, self-verifying** agent that tells a payments processor where it's **gaining or losing share** by merchant category — benchmarking its own processed volume against US Census retail sales.

| Step | What you build |
|---|---|
| 1 | Explore the benchmark (`MARKET_SHARE_BENCHMARK`) |
| 2 | Query the **semantic view** (Cortex Analyst) |
| 3 | Build a verified **fact pack** |
| 4 | **Grounded draft** under DATA RULES |
| 5 | **Verify** claims vs evidence |
| 6 | Persist the note |

**Prereq:** run `sql/setup.sql` (synthetic market, no listing), optionally followed by `sql/setup_census.sql` (US Census market). Use the role `MARKET_SHARE_ROLE`, which setup creates and which owns `SNOWPACK_MARKET_SHARE.CORE`.

In [ ]:
SCHEMA = "SNOWPACK_MARKET_SHARE.CORE"
MODEL  = "claude-sonnet-4-5"
import json
from snowflake.snowpark.context import get_active_session
session = get_active_session()
mode = session.sql(f"select mode, note from {SCHEMA}.MARKET_SHARE_DATA_SOURCE_INFO").collect()[0]
print("data mode:", mode[0], "—", mode[1])

## Step 1 — Explore the benchmark

`MARKET_SHARE_BENCHMARK` = processor volume vs market retail sales by category/month, with share in basis points (100 bps = 1%).

In [ ]:
session.sql(f"""
  SELECT category_name, month_end,
         ROUND(processed_volume_usd/1e6,1) AS processed_m,
         ROUND(market_sales_usd/1e9,1) AS market_b, share_bps
  FROM {SCHEMA}.MARKET_SHARE_BENCHMARK
  WHERE month_end = (SELECT MAX(month_end) FROM {SCHEMA}.MARKET_SHARE_BENCHMARK)
  ORDER BY share_bps DESC
""").to_pandas()

## Step 2 — Cortex Analyst over the semantic view

`MARKET_SHARE_SV` lets Analyst answer NL questions like *"which categories gained share in 2026?"*. Direct `SEMANTIC_VIEW()` query below; the agent writes this for you from plain English.

In [ ]:
session.sql(f"""
  SELECT * FROM SEMANTIC_VIEW({SCHEMA}.MARKET_SHARE_SV
      METRICS avg_share_bps, total_processed_usd
      DIMENSIONS category_name, report_year)
  WHERE report_year = 2026 ORDER BY avg_share_bps DESC
""").to_pandas()

## Step 3 — Verified fact pack

Start-of-window and latest snapshots per category — the agent's only allowed numbers.

In [ ]:
fact_pack = session.sql(f"""
  WITH w AS (SELECT MIN(month_end) a, MAX(month_end) z FROM {SCHEMA}.MARKET_SHARE_BENCHMARK)
  SELECT ARRAY_AGG(OBJECT_CONSTRUCT(
           'category', s.category_name, 'month', s.month_end::string,
           'processed_volume_usd', s.processed_volume_usd,
           'market_sales_usd', s.market_sales_usd, 'share_bps', s.share_bps))
  FROM {SCHEMA}.MARKET_SHARE_BENCHMARK s JOIN w ON s.month_end IN (w.a, w.z)
""").collect()[0][0]
print(fact_pack[:700], '...')

## Step 4 — Grounded draft (DATA RULES)

The model may use ONLY the fact pack — no invented numbers, no hedging.

In [ ]:
DATA_RULES = (
    "You are a payments-portfolio analyst. Write a concise market-share note for the processor using ONLY the FACTS JSON.\n"
    "DATA RULES: use only numbers in FACTS, copy exactly; share_bps is basis points (100 bps = 1%); dollar fields are USD, express in $B/$M; "
    "compute share change as (latest bps - earliest bps) per category from FACTS; no estimates, no hedging words, no outside benchmarks; missing = n/d.\n"
    "OUTPUT markdown starting with '# ', a one-line takeaway (which categories gained/lost share), a table (category | processed | market | share bps start->now | delta bps), then 3 bullets, ending with '## Data Coverage' (source + window)."
)
prompt = DATA_RULES + f"\n\nFACTS: {fact_pack}"
draft_md = session.sql("SELECT AI_COMPLETE(?, ?)", params=[MODEL, prompt]).collect()[0][0]
print(draft_md)

## Step 5 — Verify against the evidence

A second pass fact-checks every number against the fact pack and flags hedges — the quality gate.

In [ ]:
verify_prompt = (
    "You are a fact-checker. Compare the DRAFT against the FACTS JSON (the only allowed source). "
    "Return ONLY a valid JSON object (double quotes, no code fences, no prose) with keys: status (\"verified\" if every number "
    "and claim is supported by FACTS and there are no hedge words like likely/approximately/around/could, else \"needs_review\"), "
    "residual_hedges (integer), edits (array of short strings naming each unsupported or hedged claim).\n\n"
    f"FACTS: {fact_pack}\n\nDRAFT:\n{draft_md}"
)
raw = session.sql("SELECT AI_COMPLETE(?, ?)", params=[MODEL, verify_prompt]).collect()[0][0]
import re as _re
def _parse_verdict(x):
    t = x.strip()
    for _ in range(3):
        m = _re.match(r"^```[a-zA-Z]*\s*(.*?)\s*```$", t, _re.S)
        if m:
            t = m.group(1).strip()
        try:
            v = json.loads(t)
        except Exception:
            a, b = t.find("{"), t.rfind("}")
            if a >= 0 and b > a:
                try:
                    v = json.loads(t[a:b + 1])
                except Exception:
                    return None
            else:
                return None
        if isinstance(v, str):
            t = v.strip()
            continue
        return v
    return None
verdict = _parse_verdict(raw) or {"status": "needs_review", "residual_hedges": None, "edits": ["unparseable verifier output"]}
status = verdict.get("status", "needs_review")
edits = verdict.get("edits", []) or []
hedges = verdict.get("residual_hedges")
print(f"STATUS: {status.upper()}   residual_hedges={hedges}   edits={len(edits)}")
for e in edits:
    print(" -", e)

## Step 6 — Persist the note

In [ ]:
session.sql(
    f"INSERT INTO {SCHEMA}.MARKET_SHARE_REPORTS (subject, draft_md, report_md, status, edits_applied, residual_hedges) SELECT ?, ?, ?, ?, ?, ?",
    params=["Portfolio market share", draft_md, draft_md, status, len(edits), hedges]
).collect()
session.sql(f"SELECT run_id, subject, status, edits_applied, residual_hedges, created_at FROM {SCHEMA}.MARKET_SHARE_REPORTS ORDER BY created_at DESC LIMIT 5").to_pandas()

## Recap & extensions

You built the verified market-share loop on free data. `sql/setup.sql` already created a **`MARKET_SHARE_AGENT`** — chat with it in Snowsight → AI & ML → Agents (*"which categories are gaining share in 2026?"*).

**Next:** point `MARKET_SHARE_PORTFOLIO_MONTHLY` at your own settlement data to benchmark your real book against the market. This pack is one of a set of payments examples that reuse this exact pattern.

**Try breaking the verifier:** add a made-up forecast (e.g., *"likely to grow 10% next quarter"*) to the draft and watch Step 5 flag it *Needs review*.